# Tutorial 3: Integrating Custom Features — the Avalanche Toolbox

EEGDash's decorator system is not limited to simple functions that compute a scalar per channel. In this tutorial you will learn how to wrap a complete **multi-step scientific pipeline** — avalanche analysis — into EEGDash's feature extraction infrastructure.

**What you will learn:**

- What neural avalanches are and why they matter.
- The difference between a *preprocessor* and a *feature* in EEGDash.
- How to handle the 3-D batch convention `(batch, channels, time)` when adapting existing research code.
- How to chain three preprocessors in a nested `FeatureExtractor`.
- How `@metadata_preprocessor` lets preprocessors communicate metadata (e.g. bin size) to downstream functions.
- Why some metrics — like the Deviation from Criticality Coefficient — must be computed as *post-processing* rather than as EEGDash features.

**Prerequisites:** Tutorial 2 (custom univariate features and the decorator contract).

In [ ]:
# Standard scientific stack
import sys
import warnings
from functools import partial

import matplotlib.pyplot as plt
import numpy as np
import scipy.ndimage as ndi
from scipy.optimize import minimize_scalar
from scipy.stats import median_abs_deviation

# EEGDash
from eegdash.features import (
    FeatureExtractor,
    extract_features,
    feature_predecessor,
    metadata_preprocessor,
    multivariate_feature,
    preprocessor_output_type,
)
from eegdash.features.output_types import BasePreprocessorOutputType

print(f"Python {sys.version.split()[0]}")
try:
    import eegdash
    print(f"eegdash {eegdash.__version__}")
except Exception:
    pass
np.random.seed(42)

---
## 0. What Are Neural Avalanches?

The brain operates near a **critical point** — a transition between order and disorder that is thought to maximise dynamic range, information transmission, and storage capacity. One of the most striking signatures of criticality is **neuronal avalanches**: cascades of neural activity whose sizes and durations follow *power-law* distributions over many orders of magnitude.

### Key quantities

| Symbol | Name | Definition |
|--------|------|------------|
| σ | Branching parameter | Average number of events triggered by one event in the next time bin |
| α | Size exponent | Power-law slope of the avalanche *size* distribution |
| τ | Duration exponent | Power-law slope of the avalanche *duration* distribution |
| γ | Scaling exponent | Slope of the size–duration scaling relationship (mean size ∝ duration^γ) |
| DCC | Deviation from Criticality | \| γ_obs − (τ−1)/(α−1) \|; near 0 at criticality |

At criticality: σ ≈ 1, α ≈ 1.5, τ ≈ 2.0, γ ≈ 2.0, and DCC ≈ 0.

> **Reference:** Beggs & Plenz, *Neuronal Avalanches in Neocortical Circuits*, J. Neurosci. 23(35), 11167–11177 (2003).

---
## 1. A Single-Recording Demo

Before touching EEGDash, let's run the original avalanche pipeline on **one recording** to understand what each step does. The data below is synthetic; replace it with a real recording once you have chosen a dataset.

In [ ]:
# ── TODO: replace with a real EEG recording ───────────────────────────────────
# ds = EEGDashDataset(dataset="<dataset_id>", subject="<subject_id>", cache_dir="./data")
# raw = ds.datasets[0].raw
# data = raw.get_data()  # shape (channels, time)
# fs = raw.info["sfreq"]
# ─────────────────────────────────────────────────────────────────────────────

# Synthetic stand-in: 32 channels, 30 seconds @ 256 Hz
# We inject occasional bursts so the avalanche detector has something to find.
fs = 256.0
n_ch, n_t = 32, int(30 * fs)
rng = np.random.default_rng(0)
data = rng.standard_normal((n_ch, n_t)) * 0.5  # background noise

# Add 40 random bursts: random channels, random times, amplitude 5σ
for _ in range(40):
    ch = rng.integers(0, n_ch, size=rng.integers(2, 8))
    t0 = rng.integers(0, n_t - 20)
    data[np.ix_(ch, np.arange(t0, t0 + 10))] += rng.standard_normal() * 5

print(f"Recording: {n_ch} channels × {n_t} samples  (fs = {fs} Hz, {n_t/fs:.0f} s)")

In [ ]:
# ── Step 1: Detect avalanche events ──────────────────────────────────────────
# For each channel, z-score the signal and find local absolute-value peaks
# that exceed a threshold of k standard deviations.  The peak of each
# connected run of threshold-crossings is marked as a single event (1),
# all other samples are 0.

EPSILON = 1e-10

def _avalanche_preprocessor_2d(data, k=3.0, thresholding="std"):
    """Original (2-D) avalanche event detector.  Input: (channels, time)."""
    if thresholding == "std":
        mu    = data.mean(axis=1, keepdims=True)
        sigma = data.std(axis=1,  keepdims=True)
        sigma[sigma == 0] = EPSILON
        abs_data = np.abs((data - mu) / sigma)
    elif thresholding == "mad":
        med       = np.median(data, axis=1, keepdims=True)
        robust_sd = median_abs_deviation(data, axis=1, scale="normal", keepdims=True)
        robust_sd[robust_sd == 0] = EPSILON
        abs_data  = np.abs((data - med) / robust_sd)
    else:
        raise ValueError(f"Unknown thresholding: {thresholding!r}")

    mask = (abs_data > k).astype(np.uint8)
    # Connect threshold crossings only along the time axis
    structure = np.array([[0, 0, 0],
                          [1, 1, 1],
                          [0, 0, 0]])
    labels, n_components = ndi.label(mask, structure=structure)
    binary = np.zeros_like(data, dtype=np.uint8)
    if n_components:
        positions = ndi.maximum_position(abs_data, labels, np.arange(1, n_components + 1))
        rows, cols = zip(*positions)
        binary[rows, cols] = 1
    return binary

binary_data = _avalanche_preprocessor_2d(data, k=3.0)
print(f"Binary data shape:    {binary_data.shape}")
print(f"Total detected peaks: {binary_data.sum()}")

In [ ]:
# ── Step 2: Bin the activity ──────────────────────────────────────────────────
# Sum across channels to get *network activity* per sample, then group
# samples into non-overlapping time bins.  The bin size determines the
# temporal resolution — typically the inter-electrode distance / signal speed
# or 1–2 × the average inter-event interval.

def _bin_avalanches_2d(binary_data, fs, bin_size_sec=None, bin_size_samples=None, n_bins=None):
    """Original (2-D) binner.  Input: (channels, time)."""
    preds = [bin_size_sec is not None, bin_size_samples is not None, n_bins is not None]
    if sum(preds) != 1:
        raise ValueError("Specify exactly one of bin_size_sec, bin_size_samples, n_bins.")
    if bin_size_sec:       bin_size_samples = int(np.floor(bin_size_sec * fs))
    elif n_bins:           bin_size_samples = binary_data.shape[1] // n_bins
    if bin_size_samples <= 1:
        raise ValueError("Bin size too small.")
    network = binary_data.sum(axis=0)          # (time,)
    actual_n_bins = len(network) // bin_size_samples
    trimmed       = network[: actual_n_bins * bin_size_samples]
    binned        = trimmed.reshape(actual_n_bins, bin_size_samples).sum(axis=1)
    return binned, bin_size_samples / fs, actual_n_bins

binned_data, bin_size_sec, n_bins = _bin_avalanches_2d(binary_data, fs, bin_size_sec=1/fs * 10)  # 10-sample bins
print(f"Binned activity shape: {binned_data.shape}  ({n_bins} bins, {bin_size_sec*1000:.1f} ms each)")

In [ ]:
# ── Step 3: Detect avalanche start/end indices ────────────────────────────────
# A contiguous run of active bins (activity > 0) is one avalanche.
# Edge avalanches (touching the first or last bin) are discarded.

def _detect_avalanches_2d(binned, n_bins):
    """Original (2-D) index detector.  Input: 1-D binned activity."""
    is_active = (binned > 0).astype(int)
    diffs = np.diff(is_active, prepend=0, append=0)
    starts = np.where(diffs ==  1)[0]
    ends   = np.where(diffs == -1)[0] - 1
    if len(starts) and starts[0] == 0:           starts, ends = starts[1:], ends[1:]
    if len(ends)   and ends[-1]   == n_bins - 1: starts, ends = starts[:-1], ends[:-1]
    return starts, ends

starts, ends = _detect_avalanches_2d(binned_data, n_bins)
print(f"Detected {len(starts)} avalanches")

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)

# Top: binary event raster
ax = axes[0]
chs, ts = np.where(binary_data)
ax.scatter(ts / fs, chs, s=2, c="steelblue", alpha=0.5, linewidths=0)
ax.set_ylabel("Channel")
ax.set_title("Avalanche Events (binary raster)")
ax.set_ylim(-1, n_ch)

# Bottom: binned network activity + detected avalanches
ax = axes[1]
bin_times = np.arange(n_bins) * bin_size_sec
ax.plot(bin_times, binned_data, color="steelblue", lw=1, label="Network activity")
for s, e in zip(starts, ends):
    ax.axvspan(s * bin_size_sec, (e + 1) * bin_size_sec, color="salmon", alpha=0.4)
ax.set_xlabel("Time (s)")
ax.set_ylabel("Active channels / bin")
ax.set_title("Binned Network Activity (salmon = detected avalanches)")
ax.legend()

plt.tight_layout()
plt.show()
print(f"Bin duration: {bin_size_sec*1000:.1f} ms   |   Avalanche count: {len(starts)}")

In [ ]:
# ── Compute branching parameter (σ) by hand ───────────────────────────────────
n_avalanches = len(starts)
if n_avalanches > 0:
    n_a = binned_data[starts].astype(float)      # activity at avalanche onset
    n_d = binned_data[starts + 1].astype(float)  # activity one bin later
    with np.errstate(divide="ignore", invalid="ignore"):
        sigma = float(np.nanmean(n_d / n_a))
else:
    sigma = np.nan

# ── Compute sizes and durations ───────────────────────────────────────────────
if n_avalanches > 0:
    sizes     = np.array([binned_data[s:e+1].sum() for s, e in zip(starts, ends)])
    durations = (ends - starts + 1).astype(int)
else:
    sizes = durations = np.array([])

print(f"Branching parameter σ = {sigma:.3f}  (critical value ≈ 1.0)")
if n_avalanches > 0:
    print(f"Avalanche sizes:    min={sizes.min():.0f}  max={sizes.max():.0f}  mean={sizes.mean():.1f}")
    print(f"Avalanche duration: min={durations.min()}   max={durations.max()}   mean={durations.mean():.1f} bins")

---
## 2. The Scaling Problem

We just computed avalanche statistics for **one** recording — which required writing explicit loops, managing intermediate variables, and calling each function by hand.

Now imagine your dataset has 500 subjects × 10 sessions × 3 conditions = **15,000 recordings**.

A naive approach — a Python `for`-loop over recordings — has several problems:

| Issue | Consequence |
|-------|-------------|
| Serial execution | 15,000× slower than a parallelised pipeline |
| Manual file management | Error-prone; hard to resume after crashes |
| No batching | Cannot exploit GPU-style vectorisation across windows |

EEGDash's feature extraction pipeline solves all three. You **declare** the pipeline once as a `FeatureExtractor` and call `extract_features(windows_ds, extractor, n_jobs=-1)` — it handles batching, parallelisation, and result accumulation automatically.

The only requirement: each step in the pipeline must follow the EEGDash **decorator contract**.

---
## 3. Methodological Framing

Before adapting the avalanche code, it helps to be clear about two conceptual distinctions that the decorator system encodes.

### 3a. Preprocessor vs Feature

| | Preprocessor | Feature |
|--|--|--|
| **Returns** | Transformed data (arrays, tuples of arrays) | Scalar or vector summary per window |
| **Decorators** | `@feature_predecessor(...)`, optionally `@preprocessor_output_type(T)` | `@feature_predecessor(...)` **+** `@univariate_feature` / `@multivariate_feature` |
| **In `FeatureExtractor`** | Passed as `preprocessor=` argument | Listed in the features dict |
| **Example** | `spectral_preprocessor` → `(freq, power)` | `spectral_entropy` → one value per channel |

In the avalanche pipeline:
- `avalanche_preprocessor` → *preprocessor* (binary event array, same shape as input)
- `bin_avalanches` → *preprocessor* (summed network activity)
- `detect_avalanches` → *preprocessor* (index arrays)
- `branching_parameter`, `alpha_exponent`, … → *features* (one scalar per window)

### 3b. Multivariate Features

Avalanche features are **multivariate**: they produce one scalar per *window* (not one per *channel*). EEGDash handles this with the `@multivariate_feature` decorator.

```
Input  shape: (batch, channels, time)
                        ↓  preprocessors
Output shape: (batch,)  ← multivariate — one value per window
```

Compare with univariate features (Tutorial 2), which return `(batch, channels)` — one value per channel per window.

---
## 4. Shape Fix: From 2-D to 3-D

The original avalanche functions were written for a **single recording** at a time: inputs have shape `(channels, time)`. EEGDash operates on **batches of windows**: `(batch, channels, time)`.

The fix is always the same: add a batch dimension and vectorise (or loop) over it.

In [ ]:
# ── From the original (2-D) lab code ──────────────────────────────────────────
# def avalanche_preprocessor(data, k=3.0, thresholding='std'):
#     # data shape: (channels, time)
#     mu    = data.mean(axis=1, keepdims=True)   # axis=1 is TIME in a 2-D array
#     sigma = data.std(axis=1,  keepdims=True)   # axis=1 is TIME in a 2-D array
#     abs_data = np.abs((data - mu) / sigma)
#     ...

# ── EEGDash 3-D adaptation ────────────────────────────────────────────────────
# def avalanche_preprocessor(x, /, *, k=3.0, _metadata):
#     # x shape: (batch, channels, time)
#     mu    = x.mean(axis=-1, keepdims=True)   # axis=-1 is always TIME (last axis)
#     sigma = x.std(axis=-1,  keepdims=True)   # axis=-2 would be CHANNELS
#     abs_data = np.abs((x - mu) / sigma)
#     ...

# Rule of thumb:
#   axis=-1  →  time      (last axis)
#   axis=-2  →  channels  (second-to-last axis)
#
# Always prefer -1 / -2 over hardcoded 2 / 1.
# If the batch dimension is ever squeezed or added, negative indices stay correct;
# a hardcoded axis=2 would silently operate on the wrong dimension.

# Quick verification with a toy batch
x_toy = np.random.randn(4, 32, 512)   # 4 windows, 32 channels, 512 samples

mu_wrong = x_toy.mean(axis=-2, keepdims=True)  # collapses channels, not time
mu_right  = x_toy.mean(axis=-1, keepdims=True) # collapses time ← correct
print(f"Wrong axis=-2 (channel mean) shape: {mu_wrong.shape}   ← (batch, 1, time)")
print(f"Right axis=-1 (time mean)    shape: {mu_right.shape}   ← (batch, channels, 1)")

---
## 5. Correct Decorators — the EEGDash Interface

The current API uses **lowercase function decorators** imported from `eegdash.features`. The table below covers every decorator used in the avalanche pipeline.

### Decorator cheat-sheet

| Decorator | Position | Purpose |
|-----------|----------|---------|
| `@feature_predecessor(fn)` | Outermost | Declares what the function's input is |
| `@metadata_preprocessor` | 2nd (preprocessors only) | Required when the function **writes to** `_metadata` (adds or modifies keys). Any function can **read** `_metadata` without this decorator — only *changing* it requires it. Must return `(*data, _metadata)`. |
| `@preprocessor_output_type(T)` | 3rd | Declares the output type class |
| `@multivariate_feature` | 2nd (features only) | One scalar per *window* |
| `@univariate_feature` | 2nd (features only) | One scalar per *channel* per window |

In [ ]:
# ── Custom output types ───────────────────────────────────────────────────────
# Each preprocessor stage produces a *different* data structure.
# We create a thin subclass of BasePreprocessorOutputType for each stage.
# These classes carry no extra logic — they exist solely as type tokens
# that the FeatureExtractor uses to validate the execution graph.

class AvalancheEventsType(BasePreprocessorOutputType):
    """Binary peak array: (batch, channels, time) uint8."""

class BinnedAvalancheType(BasePreprocessorOutputType):
    """Summed network activity: (batch, n_bins) float."""

class AvalancheIndicesType(BasePreprocessorOutputType):
    """Padded index arrays: (binned_array, starts, ends, counts)."""

In [ ]:
# ── Preprocessor 1: raw signal → binary events ────────────────────────────────
# Changes from the original:
#   • axis=2 instead of axis=1 for all per-channel statistics
#   • fs is read from _metadata["info"]["sfreq"] (not a function argument)
#   • returns (binary_data, _metadata) — required by @metadata_preprocessor

@feature_predecessor()          # depends on raw signal
@metadata_preprocessor          # reads _metadata; must return (*data, _metadata)
@preprocessor_output_type(AvalancheEventsType)
def avalanche_preprocessor(x, /, *, k: float = 3.0, thresholding: str = "std", _metadata: dict):
    """Threshold and peak-pick to produce a binary event array.

    Input  shape: (batch, channels, time)
    Output shape: (batch, channels, time)  dtype uint8
    """
    if thresholding == "std":
        mu    = x.mean(axis=2, keepdims=True)
        sigma = x.std(axis=2,  keepdims=True)
        sigma[sigma == 0] = EPSILON
        abs_data = np.abs((x - mu) / sigma)
    elif thresholding == "mad":
        med       = np.median(x, axis=2, keepdims=True)
        robust_sd = median_abs_deviation(x, axis=2, scale="normal", keepdims=True)
        robust_sd[robust_sd == 0] = EPSILON
        abs_data  = np.abs((x - med) / robust_sd)
    else:
        raise ValueError(f"Unsupported thresholding: {thresholding!r}")

    mask = (abs_data > k).astype(np.uint8)
    structure = np.zeros((3, 3, 3), dtype=np.int8)
    structure[1, 1, :] = 1                      # connect along time only
    labels, n_feats = ndi.label(mask, structure=structure)
    binary = np.zeros_like(x, dtype=np.uint8)
    if n_feats:
        pos = ndi.maximum_position(abs_data, labels, np.arange(1, n_feats + 1))
        epochs, rows, cols = zip(*pos)
        binary[epochs, rows, cols] = 1
    return binary, _metadata               # <-- _metadata returned unchanged here

# ── Sanity check ──────────────────────────────────────────────────────────────
x_toy = np.random.randn(4, 8, 256).astype(np.float32)
fake_meta = {"info": {"sfreq": 256.0, "ch_names": [f"ch{i}" for i in range(8)]}}
binary_toy, _ = avalanche_preprocessor(x_toy, k=2.5, _metadata=fake_meta)
print(f"Input:  {x_toy.shape}")
print(f"Output: {binary_toy.shape}  dtype={binary_toy.dtype}")
print(f"Events per window: {binary_toy.sum(axis=(1,2))}")

In [ ]:
# ── Preprocessor 2: binary events → binned network activity ───────────────────
# NEW behaviour: bin_size_sec and n_bins are written into _metadata so that
# downstream features (e.g. tau_exponent with t_max_method='lab') can read them
# without extra arguments.

@feature_predecessor(avalanche_preprocessor)   # depends on stage 1
@metadata_preprocessor                          # modifies _metadata
@preprocessor_output_type(BinnedAvalancheType)
def bin_avalanches(binary_data, /, *, bin_size_samples: int = None, _metadata: dict):
    """Sum channels → network activity, then group into fixed-width bins.

    Adds ``bin_size_sec`` and ``n_bins`` to ``_metadata``.

    Input  shape: (batch, channels, time)  uint8
    Output shape: (batch, n_bins)          float
    """
    fs = _metadata["info"]["sfreq"]
    if bin_size_samples is None:
        bin_size_samples = max(1, int(fs))   # default: 1-second bins

    network = binary_data.sum(axis=1).astype(float)   # (batch, time)
    batch_size, n_t = network.shape
    actual_n_bins = n_t // bin_size_samples
    if actual_n_bins == 0:
        raise ValueError(f"bin_size_samples={bin_size_samples} exceeds window length {n_t}.")

    trimmed = network[:, : actual_n_bins * bin_size_samples]
    binned  = trimmed.reshape(batch_size, actual_n_bins, bin_size_samples).sum(axis=2)

    _metadata = {**_metadata,                          # <-- copy + add new keys
                 "bin_size_sec": bin_size_samples / fs,
                 "n_bins":       actual_n_bins}
    return binned, _metadata

binned_toy, meta_updated = bin_avalanches(binary_toy, bin_size_samples=10, _metadata=fake_meta)
print(f"Binned shape: {binned_toy.shape}")
print(f"Metadata keys added: bin_size_sec={meta_updated['bin_size_sec']:.4f} s, n_bins={meta_updated['n_bins']}")

In [ ]:
# ── Preprocessor 3: binned activity → padded index arrays ─────────────────────
# Instead of returning a Python dict (slow, hard to vectorise), we return
# *padded* numpy arrays:
#
#   starts / ends : shape (batch, max_avalanches), padding value = -1
#   counts        : shape (batch,)  — actual number of valid avalanches
#
# To read the avalanches for window i:
#   n   = counts[i]
#   s   = starts[i, :n]   # valid start indices
#   e   = ends[i, :n]     # valid end indices (inclusive)

@feature_predecessor(bin_avalanches)           # depends on stage 2
@preprocessor_output_type(AvalancheIndicesType)
def detect_avalanches(binned_array, /):
    """Detect contiguous active runs in the binned network activity.

    Edge avalanches (touching bin 0 or bin n_bins-1) are discarded.

    Input : (batch, n_bins)
    Output: (binned_array, starts, ends, counts)
    """
    batch_size, n_bins = binned_array.shape
    starts_list, ends_list = [], []
    counts = np.zeros(batch_size, dtype=np.intp)

    for i in range(batch_size):
        is_active = (binned_array[i] > 0).astype(np.int8)
        if not is_active.any():
            starts_list.append(np.empty(0, dtype=np.intp))
            ends_list.append(np.empty(0,   dtype=np.intp))
            continue
        diffs = np.diff(is_active, prepend=0, append=0)
        s = np.where(diffs ==  1)[0]
        e = np.where(diffs == -1)[0] - 1
        if len(s) and s[0] == 0:              s, e = s[1:],  e[1:]
        if len(e) and e[-1] == n_bins - 1:    s, e = s[:-1], e[:-1]
        starts_list.append(s)
        ends_list.append(e)
        counts[i] = len(s)

    pad = max(int(counts.max()), 1)
    starts_pad = np.full((batch_size, pad), -1, dtype=np.intp)
    ends_pad   = np.full((batch_size, pad), -1, dtype=np.intp)
    for i, (s, e) in enumerate(zip(starts_list, ends_list)):
        if len(s):
            starts_pad[i, : len(s)] = s
            ends_pad[i,   : len(e)] = e

    return binned_array, starts_pad, ends_pad, counts

binned_r, starts_r, ends_r, counts_r = detect_avalanches(binned_toy)
print(f"Output shapes — binned: {binned_r.shape},  starts: {starts_r.shape},  ends: {ends_r.shape},  counts: {counts_r}")

In [ ]:
# ── Feature 1: Branching parameter ───────────────────────────────────────────

@feature_predecessor(detect_avalanches)
@multivariate_feature
def branching_parameter(binned_array, starts, ends, counts, /, *, method: str = "naive"):
    """σ — average descendants per ancestor, per window.

    Returns shape (batch,)  — one scalar per window.
    """
    batch_size = binned_array.shape[0]
    sigmas = np.full(batch_size, np.nan)
    for i in range(batch_size):
        n = int(counts[i])
        if not n:
            continue
        s   = starts[i, :n]
        n_a = binned_array[i, s].astype(float)
        n_d = binned_array[i, s + 1].astype(float)
        if method == "naive":
            with np.errstate(divide="ignore", invalid="ignore"):
                sigmas[i] = float(np.nanmean(n_d / n_a))
        elif method == "weighted":
            total_a = n_a.sum()
            if total_a:
                sigmas[i] = n_d.sum() / total_a
        else:
            raise ValueError(f"Unknown method: {method!r}")
    return sigmas

# ── Feature 2: Alpha exponent (size distribution) ─────────────────────────────

def _nll(exp, x_logs, sum_log_data, n):
    if exp <= 1: return np.inf
    Z = np.sum(np.exp(-exp * x_logs))
    return np.inf if (Z <= 0 or not np.isfinite(Z)) else exp * sum_log_data + n * np.log(Z)

def _fit_power_law(data, system_size, n_min=100, xmin=1):
    """Truncated MLE power-law fit with KS-based cutoff selection."""
    data = np.sort(data[np.isfinite(data) & (data >= 1)])
    if data.size < n_min or system_size <= 0:
        return {"exponent": np.nan, "ks": np.nan}
    maxdata = data[-1]
    cutoffs = [c for c in range(int(0.8*system_size), int(1.5*system_size)+1) if c <= maxdata]
    if not cutoffs: cutoffs = [int(maxdata)]
    start_idx   = np.searchsorted(data, xmin, "left")
    log_cumsum  = np.insert(np.cumsum(np.log(data)), 0, 0.0)
    best = {"exponent": np.nan, "ks": np.inf}
    for cutoff in cutoffs:
        end_idx = np.searchsorted(data, cutoff, "right")
        n = end_idx - start_idx
        if n < n_min: continue
        xlogs = np.log(np.arange(xmin, cutoff + 1))
        res   = minimize_scalar(_nll, bounds=(1.0001, 10), method="bounded",
                                args=(xlogs, log_cumsum[end_idx] - log_cumsum[start_idx], n))
        pdf   = np.exp(-res.x * xlogs); pdf /= pdf.sum(); cdf = np.cumsum(pdf)
        emp   = np.searchsorted(data[start_idx:end_idx], np.arange(xmin, cutoff+1), "right") / n
        ks    = np.max(np.abs(emp - cdf))
        if ks < best["ks"]: best = {"exponent": float(res.x), "ks": ks, "cutoff": cutoff}
    return best

@feature_predecessor(detect_avalanches)
@multivariate_feature
def alpha_exponent(binned_array, starts, ends, counts, /,
                   *, n_channels: int = None, ks_threshold: float = 0.1, _metadata: dict = None):
    """α — power-law exponent of avalanche size distribution. Returns shape (batch,)."""
    if n_channels is None and _metadata is not None:
        n_channels = len(_metadata["info"]["ch_names"])
    batch_size = binned_array.shape[0]
    alphas = np.full(batch_size, np.nan)
    for i in range(batch_size):
        n = int(counts[i])
        if not n: continue
        s = starts[i, :n]; e = ends[i, :n]
        sizes = np.array([binned_array[i, s[j]:e[j]+1].sum() for j in range(n)])
        sys_sz = n_channels if n_channels else max(1, int(sizes.max()))
        fit = _fit_power_law(sizes, system_size=sys_sz)
        if not np.isnan(fit["ks"]) and fit["ks"] <= ks_threshold:
            alphas[i] = fit["exponent"]
    return alphas

@feature_predecessor(detect_avalanches)
@multivariate_feature
def tau_exponent(binned_array, starts, ends, counts, /,
                 *, ks_threshold: float = 0.1):
    """τ — power-law exponent of avalanche duration distribution. Returns shape (batch,)."""
    batch_size = binned_array.shape[0]
    taus = np.full(batch_size, np.nan)
    for i in range(batch_size):
        n = int(counts[i])
        if not n: continue
        s = starts[i, :n]; e = ends[i, :n]
        durs = (e - s + 1).astype(int)
        t_max = int(durs.max())
        fit = _fit_power_law(durs, system_size=t_max)
        if not np.isnan(fit["ks"]) and fit["ks"] <= ks_threshold:
            taus[i] = fit["exponent"]
    return taus

@feature_predecessor(detect_avalanches)
@multivariate_feature
def gamma_exponent(binned_array, starts, ends, counts, /, *, min_unique_durations: int = 3):
    """γ — size–duration scaling exponent. Returns shape (batch,)."""
    batch_size = binned_array.shape[0]
    gammas = np.full(batch_size, np.nan)
    for i in range(batch_size):
        n = int(counts[i])
        if not n: continue
        s = starts[i, :n]; e = ends[i, :n]
        sizes = np.array([binned_array[i, s[j]:e[j]+1].sum() for j in range(n)])
        durs  = (e - s + 1).astype(int)
        u_durs = np.unique(durs)
        if len(u_durs) < min_unique_durations: continue
        avg_sz = np.array([sizes[durs == t].mean() for t in u_durs])
        if (avg_sz <= 0).any(): continue
        gammas[i] = np.polyfit(np.log10(u_durs.astype(float)), np.log10(avg_sz), 1)[0]
    return gammas

print("All feature functions defined.")

In [ ]:
# ── Unit test on a toy batch ──────────────────────────────────────────────────
# Inject structured bursts so we have detectable avalanches.
rng2 = np.random.default_rng(1)
x_test = rng2.standard_normal((4, 8, 512)).astype(np.float32) * 0.3
for _ in range(25):
    ch = rng2.integers(0, 8, size=rng2.integers(2, 6))
    t0 = rng2.integers(10, 490)
    x_test[np.ix_(np.arange(4), ch, np.arange(t0, t0+8))] += rng2.standard_normal() * 6

fake_meta_test = {"info": {"sfreq": 256.0, "ch_names": [f"ch{i}" for i in range(8)]},
                  "batch_size": 4}

binary_test, m2 = avalanche_preprocessor(x_test, k=2.5, _metadata=fake_meta_test)
binned_test, m3 = bin_avalanches(binary_test, bin_size_samples=8, _metadata=m2)
b_out, s_out, e_out, c_out = detect_avalanches(binned_test)

print("Shape checks:")
print(f"  binary_test : {binary_test.shape}  dtype={binary_test.dtype}")
print(f"  binned_test : {binned_test.shape}")
print(f"  starts      : {s_out.shape}")
print(f"  ends        : {e_out.shape}")
print(f"  counts      : {c_out}  (avalanches per window)")

sigmas = branching_parameter(b_out, s_out, e_out, c_out)
alphas = alpha_exponent(b_out, s_out, e_out, c_out, _metadata=m3)
print(f"\nBranching parameters : {np.round(sigmas, 3)}")
print(f"Alpha exponents      : {np.round(alphas, 3)}  (nan = fit failed / too few avalanches)")

---
## 6. Full EEGDash Pipeline

Now we assemble a `FeatureExtractor` that runs the entire avalanche pipeline — three preprocessing stages followed by four feature functions — in a single declarative object.

### Why nested `FeatureExtractor`s?

Each `FeatureExtractor` supports exactly **one** preprocessor per level. To chain three preprocessors, we nest three `FeatureExtractor`s:

```
FeatureExtractor(preprocessor=avalanche_preprocessor)
└── FeatureExtractor(preprocessor=bin_avalanches)
    └── FeatureExtractor(preprocessor=detect_avalanches)
        ├── branching_parameter
        ├── alpha_exponent
        ├── tau_exponent
        └── gamma_exponent
```

The integer keys (`0`) at the outer levels are not prepended to the column names — only non-empty string keys produce name prefixes. The final DataFrame columns are therefore simply `branching`, `alpha`, `tau`, `gamma`.

In [ ]:
# ── Build the FeatureExtractor ────────────────────────────────────────────────
# Parameters:
#   k               : threshold multiplier (3 = 3σ, stricter → fewer avalanches)
#   bin_size_samples: bin width in samples (10 samples @ 256 Hz ≈ 39 ms)
#   n_channels      : number of EEG channels (for alpha exponent system size)

n_channels     = 8           # TODO: set to the actual channel count in your dataset
bin_sz_samples = 10          # TODO: adjust to your dataset's sampling frequency

# Why partial()?
# FeatureExtractor calls each preprocessor and feature with only the data
# arguments (plus _metadata where relevant) — there is no mechanism to pass
# extra keyword arguments at call time. functools.partial pre-binds fixed
# parameters (k, bin_size_samples, n_channels) into the callable so the
# pipeline sees a function with the right signature.

extractor = FeatureExtractor(
    preprocessor=partial(avalanche_preprocessor, k=3.0),
    feature_extractors={
        0: FeatureExtractor(
            preprocessor=partial(bin_avalanches, bin_size_samples=bin_sz_samples),
            feature_extractors={
                0: FeatureExtractor(
                    preprocessor=detect_avalanches,
                    feature_extractors={
                        "branching": branching_parameter,
                        "alpha":     partial(alpha_exponent, n_channels=n_channels),
                        "tau":       tau_exponent,
                        "gamma":     gamma_exponent,
                    },
                )
            },
        )
    },
)

print("FeatureExtractor built successfully.")

In [ ]:
# ── Create a mock windowed dataset ────────────────────────────────────────────
# TODO: replace with your real dataset loaded via EEGDashDataset + braindecode windowing.
#
# from eegdash import EEGDashDataset
# from braindecode.preprocessing import Preprocessor, preprocess, create_fixed_length_windows
# ds = EEGDashDataset(dataset="<id>", subject=None, cache_dir="./data")
# preprocess(ds, [Preprocessor("resample", sfreq=256)], n_jobs=-1)
# windows_ds = create_fixed_length_windows(ds, window_size_samples=1024, window_stride_samples=512)

from braindecode.datasets import create_from_X_y, BaseConcatDataset

rng3 = np.random.default_rng(2)
n_rec, n_win, n_ch_ds, n_t_ds = 5, 20, 8, 512
datasets = []
for rec in range(n_rec):
    X = rng3.standard_normal((n_win, n_ch_ds, n_t_ds)).astype(np.float32) * 0.5
    # Inject random bursts to ensure detectable avalanches
    for _ in range(15):
        ch_idx = rng3.integers(0, n_ch_ds, size=rng3.integers(2, 5))
        t0 = rng3.integers(10, n_t_ds - 20)
        X[np.ix_(np.arange(n_win), ch_idx, np.arange(t0, t0+8))] += rng3.standard_normal() * 5
    y  = rng3.integers(0, 2, size=n_win).astype(np.int64)    # fake binary labels
    ds = create_from_X_y(
        X, y,
        drop_last_window=False,
        sfreq=256,
        ch_names=[f"ch{i}" for i in range(n_ch_ds)],
    )
    datasets.append(ds)

windows_ds = BaseConcatDataset(datasets)
print(f"Mock dataset: {n_rec} recordings × {n_win} windows = {len(windows_ds)} total windows")
print(f"Window shape: ({n_ch_ds} channels, {n_t_ds} samples)  @ 256 Hz")

In [ ]:
# ── Run feature extraction ────────────────────────────────────────────────────
# batch_size: number of windows processed per call (tune to your RAM)
# n_jobs: -1 uses all CPU cores
with warnings.catch_warnings():
    warnings.simplefilter("ignore")  # suppress scipy power-law fitting warnings on small toy data
    features_ds = extract_features(windows_ds, extractor, batch_size=20, n_jobs=1)

print(f"Extraction complete. {len(features_ds.datasets)} recording(s) in the output.")

In [ ]:
# ── Inspect the output ────────────────────────────────────────────────────────
rec0 = features_ds.datasets[0]
print(f"Feature matrix shape:  {rec0.features.shape}")
print(f"Columns: {list(rec0.features.columns)}")
print()
print(rec0.features.describe().round(3))

---
## 7. DCC — When Features Must Depend on Other Features

The Deviation from Criticality Coefficient requires already-computed values of α, τ, and γ:

$$\text{DCC} = \left| \gamma_{\text{obs}} - \frac{\tau - 1}{\alpha - 1} \right|$$

This is a **feature-to-feature dependency** — something the current EEGDash API does not support. Each node in the execution graph can have exactly **one predecessor** (either raw signal or one preprocessor stage). A feature cannot declare two nodes as its predecessors, and a feature node cannot receive the output of another feature node as its input.

**Solution:** treat DCC as a *post-processing step* — a plain Python/NumPy function called on the extracted DataFrame after `extract_features` completes.

In [ ]:
def dcc(alphas: np.ndarray, taus: np.ndarray, gammas: np.ndarray) -> np.ndarray:
    """Deviation from Criticality Coefficient.

    DCC = |gamma_obs - (tau-1)/(alpha-1)|;  near 0 at criticality.

    Note: NOT an EEGDash feature — call this after extract_features.
    """
    with np.errstate(divide="ignore", invalid="ignore"):
        gamma_pred = (taus - 1.0) / (alphas - 1.0)
        return np.abs(gammas - gamma_pred)

# ── Compute DCC from the extracted DataFrame ───────────────────────────────────
import pandas as pd

# Concatenate all recording feature DataFrames in one step — no loop needed
full_df = pd.concat(
    [rec.features for rec in features_ds.datasets],
    ignore_index=True,
)

# Compute DCC as a vectorised column operation on the full DataFrame
full_df["dcc"] = dcc(
    full_df["alpha"].values,
    full_df["tau"].values,
    full_df["gamma"].values,
)

# For real data, attach subject-level metadata before the concat:
# for rec in features_ds.datasets:
#     rec.features["subject"] = rec.description.get("subject")  # TODO: uncomment

print(f"Full feature table shape: {full_df.shape}")
print(f"Columns: {list(full_df.columns)}")
print()
print(full_df[["branching", "alpha", "tau", "gamma", "dcc"]].describe().round(3))

---
## 8. Analysis: Group Comparison

> **TODO:** Replace the synthetic data below with your real dataset and group labels (e.g. epilepsy vs. healthy, or sleep stage).

Once you have real subjects × conditions, a typical analysis would:
1. Average features across windows per recording.
2. Compare group distributions with a Wilcoxon / Mann-Whitney U test.
3. Visualise distributions with violin plots.

In [ ]:
# ── TODO: replace with real group labels ──────────────────────────────────────
# full_df["group"] = ...  # e.g. "healthy" / "epilepsy"

# ── Synthetic group assignment for illustration ───────────────────────────────
rng4 = np.random.default_rng(3)
full_df["group"] = rng4.choice(["control", "patient"], size=len(full_df))

# Window-level → recording-level (mean per recording/group)
summary = full_df.groupby(["group"])[["branching", "alpha", "tau", "gamma", "dcc"]].mean()
print(summary.round(3))

# ── Violin plot ────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, feat in zip(axes, ["branching", "alpha", "dcc"]):
    groups = [full_df.loc[full_df["group"]==g, feat].dropna().values
               for g in ["control", "patient"]]
    vp = ax.violinplot(groups, positions=[0, 1], showmedians=True)
    for body in vp["bodies"]:
        body.set_alpha(0.6)
    ax.set_xticks([0, 1])
    ax.set_xticklabels(["control", "patient"])
    ax.set_title(feat)
    ax.set_ylabel(feat)

plt.suptitle("Avalanche Feature Distributions by Group (synthetic data)", y=1.02)
plt.tight_layout()
plt.show()
print("\n[!] Results above are on SYNTHETIC data — replace with real recordings for meaningful output.")

---
## Summary

In this tutorial you adapted a multi-step scientific pipeline to EEGDash:

1. **Shape fix** — changed all `axis=1` (2-D time axis) operations to `axis=2` (3-D batch convention).
2. **Custom output types** — created three `BasePreprocessorOutputType` subclasses as type tokens for the execution graph.
3. **Decorator contract for preprocessors:**
   ```python
   @feature_predecessor(previous_stage)   # dependency
   @metadata_preprocessor                  # reads/writes _metadata
   @preprocessor_output_type(MyType)       # output type token
   def my_preprocessor(x, /, *, _metadata): ...
   ```
4. **Metadata enrichment** — `bin_avalanches` writes `bin_size_sec` and `n_bins` into `_metadata` so downstream features can read them without extra arguments.
5. **Padded index arrays** — replaced the Python `dict` with `(binned_array, starts, ends, counts)` numpy arrays, keeping the pipeline in pure numpy.
6. **Nested `FeatureExtractor`s** — one nesting level per preprocessing stage; features sit at the innermost level.
7. **DCC as post-processing** — features that depend on other features must be computed after `extract_features`, on the resulting DataFrame.

**Decorator summary table:**

| Goal | Decorator stack (outermost → innermost) |
|------|------------------------------------------|
| Feature on raw signal | `@feature_predecessor()`, `@multivariate_feature` |
| Feature on preprocessor output | `@feature_predecessor(fn)`, `@multivariate_feature` |
| Preprocessor (no metadata) | `@feature_predecessor(fn)`, `@preprocessor_output_type(T)` |
| Preprocessor (reads/writes metadata) | `@feature_predecessor(fn)`, `@metadata_preprocessor`, `@preprocessor_output_type(T)` |

**Tutorial 4** shows how to apply the feature extraction kit to non-EEG data (e.g. calcium imaging, spike trains) by building a custom `DataLoader` that wraps any `(windows, channels, samples)` numpy array.